# OCSD – Chạy toàn bộ thực nghiệm (Run all)

Khóa luận: *Sinh ảnh cảnh nhất quán đối tượng từ phác thảo và văn bản bằng mô hình khuếch tán* – Lê Hoàng Bảo (21090331), Thái Bá Cường (21050681); GVHD: TS. Nguyễn Thanh Chuyên.

**Cập nhật 03/11/2026:** đã nhận đủ `foreground_screen` của run `20261103T035056Z_dc40e4b5`. Đã nhận review: A–D đủ số lượng/hình dáng; A/B chính xác nhất nhưng chưa tự nhiên. **Không cần chấm lại hoặc chạy lại GPU ở bước này.** [Xem A–D bằng PNG, không cần HTML](https://raw.githubusercontent.com/bobbibao/ocsd-thesis/main/docs/reviews/foreground_20261103_ABCD.png). Hướng mới: điều kiện hóa trực tiếp từng đối tượng bằng nét phác thảo với lõi InstanceDiffusion chính thức. Đây là ứng viên đang kiểm chứng, chưa có kết quả chứng minh chất lượng ảnh tốt hơn.

**Bước tiếp theo:** Runtime → Change runtime type → **Hardware accelerator: None (CPU)**. Giữ `RUN_MODE="direct_preflight"`, `RUN_ID=""` → Runtime → **Run all**. Notebook tải trọng số công khai (vài GB), kiểm tra tương thích CPU và tải về ZIP log nhỏ. Gửi ZIP vào cuộc trò chuyện; không cần export screen hay GH_TOKEN. `PASS` chưa phải kết quả chất lượng ảnh. Các ô A–F tự bỏ qua. [Chi tiết](https://github.com/bobbibao/ocsd-thesis/blob/main/docs/DIRECT_INSTANCE_PREFLIGHT.md).

Chỉ khi chủ động tái lập thí nghiệm cũ: chọn `RUN_MODE="experiment"` và GPU L4/A100. Luồng này chạy 4 giai đoạn: (A) dựng bộ dữ liệu, (B) sinh ảnh cho E1–E4 và khảo sát α, (C) đánh giá, (D) xuất bảng/hình/tóm tắt. Mọi kết quả nằm trong `DRIVE_ROOT/results/runs/<run_id>/` (`summary.md`, `summary.json`, `tables/`, `figures/`, `progress.json`). Lưu RUN_ID để resume. Chỉ tái sử dụng kết quả cùng code, cấu hình, thư viện và dữ liệu. Xem docs/EXPERIMENT_RUNBOOK.md.

In [ ]:
#@title 1. Cấu hình (chỉ sửa ô này)
RUN_MODE   = "direct_preflight"  #@param ["direct_preflight", "experiment"]
REPO_URL   = "https://github.com/bobbibao/ocsd-thesis.git"  #@param {type:"string"}
BRANCH     = "main"                                             #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/KLTN_OCSD"                 #@param {type:"string"}
TIER       = "foreground_screen"   #@param ["foreground_screen", "clean_screen", "count_screen", "instance_screen", "semantic_screen", "repair_screen", "screen", "pilot", "paper", "full"]
BACKBONE   = "sd15"    #@param ["sd15", "sd21"]
MAX_MINUTES = 20      #@param {type:"number"}
SKIP_JOBS  = ""        #@param {type:"string"}
RUN_ID     = ""        #@param {type:"string"}
PUSH_RESULTS = True    #@param {type:"boolean"}
# pilot: kiểm tra nhanh (~1-2 giờ GPU); paper: đủ số liệu cho khóa luận; full: cho bài báo (cần thêm GPU).
# Có thể bấm "Run all" nhiều lần: ảnh/độ đo đã có trên Drive sẽ được bỏ qua và chạy tiếp phần còn lại.
# SKIP_JOBS: comma-separated job prefixes to leave out, e.g. "pareto" or "pareto,E3_quickdraw_power" (GPU budget).

In [ ]:
#@title 2. Gắn Drive, kiểm tra loại runtime, lấy mã nguồn mới
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
import shutil
if RUN_MODE not in ("direct_preflight", "experiment"):
    raise ValueError("Unknown RUN_MODE")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True) if shutil.which("nvidia-smi") else None
if RUN_MODE == "direct_preflight":
    if RUN_ID:
        raise ValueError("CPU preflight uses a separate log: leave RUN_ID empty")
    if gpu is not None and gpu.returncode == 0:
        raise RuntimeError("Bước này dùng CPU: Runtime > Change runtime type > Hardware accelerator: None, rồi Run all")
    print("CPU preflight: kiểm tra trọng số thật; không sinh ảnh, không cần GH_TOKEN.")
else:
    if gpu is None or gpu.returncode:
        raise RuntimeError("Experiment mode requires a GPU runtime")
    print(gpu.stdout)
CODE = "/content/KLTN_OCSD"
if not REPO_URL:
    # chưa có repo: dùng bản mã nguồn đã tải lên Drive (thư mục DRIVE_ROOT/code)
    CODE = os.path.join(DRIVE_ROOT, "code")
elif os.path.exists(CODE):
    if not RUN_ID:
        dirty = subprocess.run(["git", "-C", CODE, "status", "--porcelain"], check=True, capture_output=True, text=True).stdout
        if dirty.strip():
            raise RuntimeError("Checkout has local changes. Preserve them; use a fresh runtime for current GitHub code, or set RUN_ID to resume.")
        subprocess.run(["git", "-C", CODE, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", CODE, "merge", "--ff-only", "FETCH_HEAD"], check=True)
    print("Using checkout (explicit RUN_ID preserves its version):", CODE)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, CODE], check=True)
print("Mã nguồn:", CODE, subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
sys.path.insert(0, CODE)
# drop modules imported by an earlier run in this kernel, so the selected code is what runs
for _m in [m for m in sys.modules if m == "ocsd" or m.startswith("ocsd.")]: del sys.modules[_m]
# HF cache on LOCAL disk: Drive cannot store the cache's symlinks (broken files -> "SafetensorError: header too large").
# Models re-download each new session (~2-4 min on Colab).
os.environ["HF_HOME"] = "/content/hf_cache"
# Public models do not require a token; optional HF authentication can use Colab secrets.

In [ ]:
#@title 3. Kiểm tra mô hình trực tiếp trên CPU và tải ZIP log
if RUN_MODE == "direct_preflight":
    import json, uuid
    from pathlib import Path
    from zipfile import ZipFile, ZIP_DEFLATED
    from google.colab import files
    DIRECT_OUT = Path(DRIVE_ROOT) / "results" / "direct_preflight" / ("attempt_" + uuid.uuid4().hex[:12])
    DIRECT_OUT.mkdir(parents=True)
    setup_log = DIRECT_OUT / "setup.log"
    print("Cài thư viện và tải trọng số công khai (vài GB); lần đầu có thể mất nhiều phút.", flush=True)
    installed = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                               os.path.join(CODE, "requirements-direct-preflight.txt")], capture_output=True, text=True)
    setup_log.write_text(installed.stdout + "\n" + installed.stderr)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="", PYTHONUNBUFFERED="1")
    checked = subprocess.run([sys.executable, os.path.join(CODE, "tools", "preflight_direct.py"),
                              "--cache", "/content/ocsd-direct-cache", "--out", str(DIRECT_OUT),
                              "--setup-log", str(setup_log), "--setup-returncode", str(installed.returncode)], env=env)
    latest = DIRECT_OUT / "latest_preflight.json"
    if latest.exists():
        result = json.loads(latest.read_text())
        archive = result["archive"]
        print("Kết quả kiểm tra:", result["status"])
    else:
        # Preserve even an OS kill/import crash; never download an older successful run.
        archive = str(DIRECT_OUT / "ocsd_direct_preflight_incomplete.zip")
        (DIRECT_OUT / "incomplete.json").write_text(json.dumps(dict(status="incomplete", returncode=checked.returncode,
            message="Process stopped before the report completed. Review logs/RAM; no GPU run is recommended."), indent=2))
        with ZipFile(archive, "w", ZIP_DEFLATED) as z:
            for p in DIRECT_OUT.rglob("*"):
                if p.is_file() and p.suffix in (".log", ".json"):
                    z.write(p, str(p.relative_to(DIRECT_OUT)))
        print("Kiểm tra chưa hoàn tất; đã giữ log để tìm lỗi.")
    print("Gửi file ZIP này vào cuộc trò chuyện:", archive)
    print("PASS chỉ xác nhận tương thích CPU. Chưa chứng minh ảnh đẹp hơn; chưa chuyển sang GPU/paper/full.")
    try:
        files.download(archive)
    except Exception as exc:
        print("Không tự tải được; lấy file theo đường dẫn Drive ở trên:", exc)
else:
    print("Experiment mode: bỏ qua CPU preflight.")

In [ ]:
#@title 3. Install libraries (keeps Colab's PyTorch), start the Drive run log
if RUN_MODE == "experiment":
    LOGS = os.path.join(DRIVE_ROOT, "results", "logs"); os.makedirs(LOGS, exist_ok=True)
    # Colab ships torchao 0.10, which peft rejects when adding LoRA ("incompatible version of torchao"); OCSD does not use it
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(CODE, "requirements-colab.txt")], capture_output=True, text=True)
    open(os.path.join(LOGS, "pip_install.log"), "w").write(r.stdout + "\n" + r.stderr)
    print(r.stdout[-2000:], r.stderr[-3000:])
    if r.returncode != 0:
        raise RuntimeError(f"pip install failed; see {LOGS}/pip_install.log")
    import importlib, ocsd; importlib.reload(ocsd)
    from ocsd import hfcache
    hfcache.repair(os.environ["HF_HOME"])   # delete broken model files so they download again
    from ocsd import runlog
    from ocsd.config import Paths, ExperimentConfig, TIERS
    P = Paths(DRIVE_ROOT); P.makedirs()
    E = ExperimentConfig(backbone=BACKBONE, tier=TIER, skip_jobs=SKIP_JOBS,
                         extra_detectors=() if TIER in ("screen", "repair_screen", "semantic_screen", "instance_screen", "count_screen", "clean_screen", "foreground_screen") else ("owlv2", "owlv2d", "detr"))
    from ocsd.provenance import start_run
    P, RUN = start_run(P, E, run_id=RUN_ID, notebook="notebooks/RUN_ALL.ipynb")
    RUN_ID = P.run_id
    runlog.start(P.results, CODE)
    LOGS = os.path.join(P.results, "logs")
    E.save(os.path.join(P.results, "experiment_config.json"))
    print("RUN_ID:", RUN_ID, "results:", P.results)
    print(E)

## Các ô dưới chỉ chạy khi RUN_MODE=experiment
Phép thử cũ `foreground_screen`: 1 cảnh xe đạp × 4 đối chứng = 4 ảnh cuối. So sánh clean cũ, anchor có nền cảnh, mask học được dạng cứng và dạng mềm. BiRefNet-lite chạy CPU, tải khoảng 224 MB một lần và kiểm tra SHA-256. Không gửi ảnh ra dịch vụ ngoài. Cùng mẫu M2, vị trí, vùng sửa và số bước diffusion; chi phí tách nền ghi riêng. Đã chạy và nhận review: người dùng ưu tiên hai nhánh tách nền, count/shape đạt trên cảnh này, nhưng chưa tự nhiên. Xem docs/FOREGROUND_20261103_RESULTS.md. Không chạy lại sweep này hoặc paper/full theo kết quả hiện tại. Chỉ dùng RUN_ID mới nếu chủ động tái lập phép thử; đây chưa phải model đạt chất lượng.

## A. Bộ dữ liệu
QuickDraw-Scenes (tổng hợp, 12 ô: số đối tượng 1/3/5/8+ × độ phức tạp đơn giản/trung bình/phức tạp) và COCO-Sketch (ảnh thật COCO val2017, phác thảo PiDiNet, mặt nạ thật). Lần đầu mất ~10–20 phút (tải COCO 1 GB).

In [ ]:
#@title A. Build benchmarks (skipped if already built for this tier)
if RUN_MODE == "experiment":
    from ocsd import stages
    with RUN.stage("A_data"):
        stages.setup_data(P, TIER, with_coco=TIER not in ("screen", "repair_screen", "semantic_screen", "instance_screen", "count_screen", "clean_screen", "foreground_screen"))
        RUN.seal_data()
        RUN.plan(E)
    from IPython.display import Image, display
    f = os.path.join(P.results, "figures", "benchmark_examples.png")
    if os.path.exists(f): display(Image(f, width=900))

## B. Sinh ảnh
Thứ tự: E3 QuickDraw (mọi baseline + OCSD-lite trên toàn bộ tập) → OCSD và Zhang et al. (cần học định danh, trên tập con phân tầng) → E3 COCO → E4 cắt bỏ → khảo sát α. `MAX_MINUTES` giới hạn thời gian của ô này; chạy lại để tiếp tục.

## T. Tuning
Paper/full tiers only, on the pilot scenes, which are then left out of the evaluation: (1) α × LoRA strength, (2) how M5(b) energy guidance is applied, (3) which of M5(a) / M5(b) to keep, jointly with α. Writes `results/tuning/tuned.json`; finished phases are skipped (phase 3 alone is about 1 A100 hour).

In [ ]:
#@title T. Tune OCSD on the pilot scenes (finished phases are skipped)
if RUN_MODE == "experiment":
    import torch
    eng = vis = None
    with RUN.stage("T_tune"):
        eng, vis = stages.tune(P, E, max_minutes=MAX_MINUTES)

In [ ]:
#@title B1. Plan and remaining images
if RUN_MODE == "experiment":
    for j in stages.experiment_plan(P, E):
        print(f"{j['name']:22s} {len(j['scenes']):4d} scenes x {len(j['methods']):2d} methods x {len(j['seeds'])} seeds -> {stages._count_todo(P, j)} images left")

In [ ]:
#@title B2. Generate images (resumable: finished images are skipped)
if RUN_MODE == "experiment":
    with RUN.stage("B_generate"):
        eng, vis = stages.generate(P, E, max_minutes=MAX_MINUTES, eng=eng, vis=vis)

## C. Đánh giá
OPR, OCE, độ chính xác đếm, mIoU, RA bằng bộ phát hiện **OWLv2** (khác Grounding DINO mà phương pháp dùng bên trong, để tránh tối ưu hóa theo chính độ đo); CLIP score toàn ảnh và cấp đối tượng; ID-Sim (DINOv2); FID/KID; LPIPS.

In [ ]:
#@title C. Metrics (only new images are evaluated; FID reused when unchanged)
if RUN_MODE == "experiment":
    if eng is not None:
        del eng
    from ocsd.methods import _GLIGEN
    _GLIGEN.clear()
    import gc
    gc.collect(); torch.cuda.empty_cache()
    with RUN.stage("C_evaluate"):
        vis = stages.evaluate(P, E, vis=vis, fid=TIER not in ("screen", "repair_screen", "semantic_screen", "instance_screen", "count_screen", "clean_screen", "foreground_screen"))

## D. Báo cáo

In [ ]:
#@title D. Tables, figures, statistics, summary
if RUN_MODE == "experiment":
    with RUN.stage("D_report"):
        summary = stages.report(P, E)
    RUN.finish(E)  # checksum only after the stage has written its final status
    from IPython.display import Markdown, display
    display(Markdown(open(os.path.join(P.results, "summary.md")).read()))
    for f in summary["figures"]:
        display(Image(f, width=900))

In [ ]:
#@title E. Run status (also saved to results/logs/status.json; errors in results/logs/errors/)
if RUN_MODE == "experiment":
    runlog.summary()
    if os.path.exists(os.path.join(LOGS, "LATEST_ERROR.txt")):
        print("\nMost recent error:\n" + open(os.path.join(LOGS, "LATEST_ERROR.txt")).read()[-3000:])

In [ ]:
#@title F. Send logs and results to GitHub (branch colab-results) so Claude can read them
if RUN_MODE == "experiment":
    from ocsd import sync
    if PUSH_RESULTS:
        sync.try_push_results(P.results, sync.colab_token())
    else:
        print("Results retained on Drive:", P.results)